**Cell #01**

# RAG11 Nutrition — Stage 3: Reranking Examples

Adds a **rerank** pass on top of the Stage 2 retrieval + generation
pipeline, and demonstrates it with 3 nutrition-specific examples.

## What reranking is, in one picture

A vector search embeds the question and every chunk *independently*, then
ranks chunks by how close their embeddings are to the question's — fast,
but only a rough proxy, because "roughly the same topic" and "actually
answers this question" are not the same thing. A **reranker** instead
looks at the question and one candidate chunk *together* and scores how
well that specific chunk answers that specific question — slower per
comparison, but far more precise. The standard pattern (and the one used
below) is: cast a wide, cheap net with vector search, then use the
reranker to pick the truly best few out of that pool.

A simple illustration before we run the real thing — say a nutrition RAG
gets asked *"How many grams of protein per kilogram does the RDA
recommend?"*:

**Step 1 — retrieval (fast, approximate).** A vector search pulls chunks
that are broadly *about protein*:

```
1. "Protein: structure, function, and food sources..."      (0.83 similarity)
2. "The RDA for protein is 0.8 g/kg body weight/day..."      (0.81 similarity)
3. "Protein quality and amino acid scoring..."               (0.80 similarity)
```

Notice: the chunk with the actual number (#2) isn't ranked first, even
though it's clearly the best match for *this* question — embedding
similarity only sees "these are all about protein," not "this one has the
number you asked for."

**Step 2 — reranking (slow, precise).** A reranker scores the question
against each candidate jointly and reorders them:

```
1. "The RDA for protein is 0.8 g/kg body weight/day..."      (rerank score: 0.95)  <- the actual answer
2. "Protein: structure, function, and food sources..."       (rerank score: 0.35)
3. "Protein quality and amino acid scoring..."                (rerank score: 0.20)
```

Now the chunk that actually answers the question is what gets sent to
Claude. The rest of this notebook runs that same two-step pattern for
real, against this project's own ingested nutrition textbooks, using
Voyage AI's `rerank-2` cross-encoder reranker (the same provider already
used for embeddings in Stage 1.2/Stage 2, so no new API account or
dependency is needed).

## Prerequisites

Same as `../stage2_ask_examples1.ipynb`: `stage1_2_eda_load_chunks.ipynb` must
already have loaded and embedded your chunks into Supabase, and `.env`
must be filled in (`cp .env.sample .env`). **No table or schema changes
are required for anything in this notebook** — see
`reusable_code/README.md` for exactly why, and what to add later only if
you want persistent, queryable manual overrides.

This notebook gets its retrieval/generation logic from `./reusable_code/`
instead of redefining it — the same package `../stage2_ask_examples1.ipynb`
now imports from, so both notebooks share one implementation of
`ask_question`, `retrieve_chunks`, etc.


In [ ]:
# Cell #02

from reusable_code import (
    init_clients,
    ask_question,
    retrieve_chunks,
    rerank_chunks,
    update_rank_value,
    page_numbers_for_chunk,
    NUM_CONTEXT_CHUNKS,
    RERANK_MODEL,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
)

clients = init_clients()
print("Clients ready.")
print("Embedding model:", EMBEDDING_MODEL, "| Rerank model:", RERANK_MODEL, "| Generation model:", GENERATION_MODEL)


**Cell #03**

## A helper to see retrieval vs. rerank side by side

`show_rerank_comparison()` runs the two-step pattern from the intro above
against the real database: fetch a wide candidate pool with
`retrieve_chunks()`, rerank it down with `rerank_chunks()`, and print both
orderings so the effect of reranking is visible rather than assumed.


In [ ]:
# Cell #04

def show_rerank_comparison(question: str, candidate_pool: int = 15, top_n: int = NUM_CONTEXT_CHUNKS,
                            rerank_model: str = RERANK_MODEL):
    """Retrieve `candidate_pool` chunks by vector search, rerank them down
    to `top_n`, and print the before/after ordering side by side. Returns
    (candidates, reranked) so the caller can inspect or reuse either list."""
    print(f"Q: {question}\n")

    candidates = retrieve_chunks(question, match_count=candidate_pool)
    print(f"-- Step 1: vector search top {len(candidates)} (cosine distance, closest first) --")
    for i, row in enumerate(candidates, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:110]
        source = row["rowJSON"].get("source_key", "?")
        print(f"  {i:>2}. dist={row['cosine_distance']:.4f}  [{source}]  {preview}...")

    reranked = rerank_chunks(question, candidates, top_n=top_n, model=rerank_model)
    print(f"\n-- Step 2: reranked top {len(reranked)} (Voyage {rerank_model} relevance score) --")
    for i, row in enumerate(reranked, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:110]
        source = row["rowJSON"].get("source_key", "?")
        moved = "(same spot)" if row["retrieval_rank"] == i else f"(was #{row['retrieval_rank']})"
        print(f"  {i:>2}. score={row['rerank_score']:.4f} {moved:<12} [{source}]  {preview}...")

    top1_changed = bool(candidates) and bool(reranked) and candidates[0]["rowGUID"] != reranked[0]["rowGUID"]
    print(f"\nTop-1 chunk changed after reranking: {top1_changed}")
    print("-" * 80)
    return candidates, reranked


example_results = {}   # question -> (candidates, reranked), filled in by the 3 examples below


**Cell #05**

## Three nutrition examples for the rerank demonstration

Adapted from the same "fast-but-rough vs. slow-but-precise" pattern shown
in the intro, each chosen for a reason a plain vector search tends to
struggle with:

1. **A precise fact buried among broadly-similar chunks** — many chunks
   are generally about protein; only one has the actual RDA number.
2. **A common oversimplification phrased as yes/no** — vector search
   tends to surface generic "caffeine"/"hydration" chunks, not necessarily
   the one that addresses *this specific claim*.
3. **A comparison question** — the answer lives in one chunk that
   contrasts two things directly, while chunks about each half of the
   comparison separately can out-rank it on pure similarity.


In [ ]:
# Cell #06

EXAMPLE_QUESTIONS = [
    # 1. Numeric fact vs. general topic chunks.
    "How many grams of protein per kilogram of body weight does the RDA recommend for an average adult?",
    # 2. Common oversimplification, yes/no framing.
    "Does drinking coffee before exercise dehydrate you enough to hurt performance?",
    # 3. Direct comparison question.
    "How does soluble fiber's effect on LDL cholesterol differ from insoluble fiber's effect?",
]


**Cell #07**

### Example 1 — a precise number among many general-topic chunks


In [ ]:
# Cell #08

example_results[EXAMPLE_QUESTIONS[0]] = show_rerank_comparison(EXAMPLE_QUESTIONS[0])


**Cell #09**

### Example 2 — a common oversimplification


In [ ]:
# Cell #10

example_results[EXAMPLE_QUESTIONS[1]] = show_rerank_comparison(EXAMPLE_QUESTIONS[1])


**Cell #11**

### Example 3 — a direct comparison question


In [ ]:
# Cell #12

example_results[EXAMPLE_QUESTIONS[2]] = show_rerank_comparison(EXAMPLE_QUESTIONS[2])


**Cell #13**

## `ask_question(..., use_rerank=...)` end to end

`use_rerank` is an optional keyword argument on `ask_question` — it
defaults to `False`, so every existing call in `../stage2_ask_examples1.ipynb`
(`ask_question(question)`) behaves exactly as before. Passing
`use_rerank=True` wires in the same over-fetch-then-rerank pattern used
above, then generates Claude's answer from the reranked chunks instead of
the raw vector-search order.


In [ ]:
# Cell #14

demo_question = EXAMPLE_QUESTIONS[0]

baseline = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_rerank=False)
reranked_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_rerank=True)

print("Q:", demo_question)

print("\n-- ask_question(..., use_rerank=False) [default] --")
print("chunks used:", baseline["chunks_used"], "| candidates considered:", baseline["candidates_considered"])
print("source pages:", baseline["source_pages"])
if baseline["short_answer"]:
    print("Short answer:", baseline["short_answer"])
print(baseline["answer"])

print("\n-- ask_question(..., use_rerank=True) --")
print("chunks used:", reranked_answer["chunks_used"], "| candidates considered:", reranked_answer["candidates_considered"])
print("rerank model:", reranked_answer["rerank_model"])
print("source pages:", reranked_answer["source_pages"])
if reranked_answer["short_answer"]:
    print("Short answer:", reranked_answer["short_answer"])
print(reranked_answer["answer"])


**Cell #15**

## Manual overrides with `update_rank_value` — no schema change required

Sometimes a human reviewer knows something the automatic ranking
doesn't — e.g. "chunk #3 has a safety caveat and must be read first."
`update_rank_value()` lets you force that, purely client-side by default:
it re-sorts the list you already retrieved/reranked, it does not require
(or create) any new Supabase column, RPC, or migration. See
`reusable_code/README.md` for exactly why (short version: `rowJSON` is
already a schemaless `jsonb` column, so this is just an in-memory
annotation unless you opt into `persist=True`).


In [ ]:
# Cell #16

_, example1_reranked = example_results[EXAMPLE_QUESTIONS[0]]

if len(example1_reranked) >= 3:
    override_target = example1_reranked[2]   # pretend a reviewer flagged the #3 chunk

    print("Before manual override:")
    for i, row in enumerate(example1_reranked, start=1):
        print(f"  {i}. {row['rowGUID'][:8]}...  score={row['rerank_score']:.4f}")

    manually_reordered = update_rank_value(
        example1_reranked,
        row_guid=override_target["rowGUID"],
        new_value=0.999,
        reason="Reviewer: this chunk has a safety caveat and must be read first.",
        # persist=True would also write this into that row's real rowJSON in
        # Supabase (an ordinary jsonb merge, no ALTER TABLE) -- left off
        # here since this is just a demo of the in-memory override.
    )

    print("\nAfter manual override (client-side only -- nothing written to Supabase):")
    for i, row in enumerate(manually_reordered, start=1):
        score = row.get("manual_rank_score", row.get("rerank_score"))
        tag = "  <- manually overridden" if "manual_rank_score" in row else ""
        print(f"  {i}. {row['rowGUID'][:8]}...  score={score:.4f}{tag}")
else:
    print("Example 1 didn't return enough chunks to demo an override -- "
          "make sure Stage 1.2 has ingested chunks into Supabase, then re-run the cell above.")


**Cell #17**

## Summary across the 3 examples

Same idea as Stage 2's summary table: a quick scan of whether reranking
actually changed the top pick for each example question, and how many raw
candidates were considered to get there.


In [ ]:
# Cell #18

print(f"{'#':<3} {'top-1 changed?':<16} {'candidates':>10}  question")
for i, question in enumerate(EXAMPLE_QUESTIONS, start=1):
    candidates, reranked = example_results[question]
    changed = bool(candidates) and bool(reranked) and candidates[0]["rowGUID"] != reranked[0]["rowGUID"]
    print(f"{i:<3} {str(changed):<16} {len(candidates):>10}  {question}")


**Cell #19**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock
recovery and conflict resolution), the same helper `../stage2_ask_examples1.ipynb`
uses -- now shared via `reusable_code.save_to_github` instead of being
redefined here.


In [ ]:
# Cell #20

from reusable_code import save_to_github

save_to_github("stage2_ask_examples2_rerank.ipynb - rerank examples added")
